# Open Router — un chatbot de consola

Mismo `POST` de siempre, pero ahora dentro de un `while True`.

- La API es **stateless**: no recuerda nada. La "memoria" es la lista `messages`.
- Cada vuelta del bucle: agregar `user` → llamar → imprimir → agregar `assistant`.

**Necesitas una API key**. Con un modelo `:free` no necesitas tarjeta.

In [ ]:
import os
import json
import time

import httpx

BASE_URL = "https://openrouter.ai/api/v1"
MODELO_FREE = "meta-llama/llama-3.3-70b-instruct:free"

# La API key es un SECRETO: no la escribas aquí ni la subas a git.
# Copia .env.example a .env y pon tu key ahí.
try:
    from dotenv import load_dotenv

    load_dotenv()
except ModuleNotFoundError:
    raise

API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert API_KEY, "Falta OPENROUTER_API_KEY"

In [ ]:
def preguntar(messages: list[dict], modelo: str = MODELO_FREE) -> str:
    """Manda la lista completa de mensajes y devuelve el texto de la respuesta."""
    r = httpx.post(
        f"{BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": modelo, "messages": messages},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

## El mismo patrón, en un bucle

`messages` arranca solo con el `system` y **crece** con la conversación.
En cada vuelta mandamos la lista completa con `preguntar()`.

Escribe `salir` o `exit` para terminar.

In [ ]:
messages = [{"role": "system", "content": "Eres un asistente breve y claro."}]

while True:
    entrada = input("tú> ") # recupera tu entrada. 
    if entrada.strip() in {"salir", "exit"}:
        break

    messages.append({"role": "user", "content": entrada})
    respuesta = preguntar(messages)          # manda la lista completa
    print("bot>", respuesta)

    messages.append({"role": "assistant", "content": respuesta})

## Mira la "memoria"

`messages` es toda la memoria que hay. Así quedó después de la conversación:

In [ ]:
print(json.dumps(messages, indent=2, ensure_ascii=False))
print(f"\n{len(messages)} turnos en la lista")